In [1]:
# -*- coding: utf-8 -*-
# NOTEBOOK : BENCHMARK DES MÉTHODES D'ÉQUILIBRAGE (SANS SMOTEENN)

# =============================================================================
# CELLULE 1 : IMPORTS ET CONFIGURATION PRINCIPALE
# =============================================================================
print("--- Cellule 1: Imports et Configuration ---")

# --- Imports ---
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, auc, average_precision_score
)

# Imbalanced-learn (pip install imbalanced-learn)
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek # SMOTEENN retiré
from imblearn.under_sampling import RandomUnderSampler

# LightGBM (pip install lightgbm)
try:
    import lightgbm as lgb
    lgbm_available = True
except ImportError:
    lgbm_available = False
    print("Warning: LightGBM non trouvé. Il sera ignoré dans les tests.")

# --- Configuration ---
DATA_PATH = "/home/kaoutar/Desktop/data/"
# !! VÉRIFIEZ BIEN CE NOM DE FICHIER !! Doit contenir les 94 features + cible
INPUT_FILE = os.path.join(DATA_PATH, 'dataset_X94_clean_enhanced.csv') # Ou le nom que vous avez utilisé

RANDOM_STATE = 42 # Pour reproductibilité
# ---------------------------------------------

print("-" * 50)
print("Chargement des Données Initiales...")
if os.path.exists(INPUT_FILE):
    try:
        df_final = pd.read_csv(INPUT_FILE, sep=',')
        print(f"Données chargées depuis {INPUT_FILE}. Shape: {df_final.shape}")

        if 'appetent' not in df_final.columns:
             raise ValueError("Colonne cible 'appetent' manquante.")

        # Préparer X (features) et y (cible)
        X_source = df_final.drop('appetent', axis=1).select_dtypes(include=[np.number])
        y_source = df_final['appetent']

        # Vérifier NaNs (normalement 0, mais sécurité)
        if X_source.isna().sum().sum() > 0:
            print(f"ATTENTION: {X_source.isna().sum().sum()} NaNs détectés dans X_source. Remplissage par 0...")
            X_source = X_source.fillna(0)
        if y_source.isna().any():
            raise ValueError("NaNs détectés dans la colonne cible y_source !")

        # Vérifier la variance (sécurité)
        print(f"Shape X_source avant filtrage variance: {X_source.shape}")
        cols_before_var_filt = X_source.columns
        variance = X_source.var()
        cols_to_keep = variance[(variance > 1e-6) & variance.notna()].index
        X_source = X_source[cols_to_keep].copy()
        cols_after_var_filt = X_source.columns
        dropped_cols_var = cols_before_var_filt.difference(cols_after_var_filt)
        if not dropped_cols_var.empty:
            print(f"  Colonnes à variance nulle/quasi-nulle supprimées: {dropped_cols_var.tolist()}")

        print(f"\nPréparation X_source/y_source OK. Shape X_source final: {X_source.shape}, Shape y_source: {y_source.shape}")
        data_loaded_successfully = True
    except Exception as e:
        print(f"ERREUR lors du chargement ou préparation initiale: {e}")
        data_loaded_successfully = False
else:
    print(f"ERREUR: Fichier {INPUT_FILE} non trouvé.")
    data_loaded_successfully = False

print("-" * 50)
print("Imports, Configuration et Chargement de Données Terminés.")
# --- FIN Cellule 1 ---

--- Cellule 1: Imports et Configuration ---


--------------------------------------------------
Chargement des Données Initiales...
Données chargées depuis /home/kaoutar/Desktop/data/dataset_X94_clean_enhanced.csv. Shape: (1051281, 95)
Shape X_source avant filtrage variance: (1051281, 94)

Préparation X_source/y_source OK. Shape X_source final: (1051281, 94), Shape y_source: (1051281,)
--------------------------------------------------
Imports, Configuration et Chargement de Données Terminés.


In [2]:
# =============================================================================
# CELLULE 2 : TEST AVEC SMOTE
# =============================================================================
if data_loaded_successfully:
    print("\n--- Test avec Méthode d'Équilibrage : SMOTE ---")
    IMBALANCE_METHOD_CURRENT_TEST = 'SMOTE'

    # 1. Séparation Train/Test
    X_train, X_test, y_train, y_test = train_test_split(X_source, y_source, test_size=0.2,
                                                        random_state=RANDOM_STATE, stratify=y_source)
    # 2. Rééquilibrage
    print(f"  Application de {IMBALANCE_METHOD_CURRENT_TEST}...")
    sampler = SMOTE(random_state=RANDOM_STATE)
    X_train_filled = X_train.fillna(X_train.median())
    X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_filled, y_train)
    print(f"  Shape Train après {IMBALANCE_METHOD_CURRENT_TEST}: {X_train_resampled.shape}")

    # 3. Scaling
    numerical_cols_to_scale = X_source.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X_source.select_dtypes(include='int').columns if X_source[col].nunique(dropna=False) <= 2]
    age_group_cols = [col for col in X_source.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in numerical_cols_to_scale if c not in cols_to_exclude_from_scaling and c in X_train_resampled.columns]
    X_train_scaled = X_train_resampled.copy()
    X_test_scaled = X_test.copy()
    if numerical_cols_to_scale:
         scaler = StandardScaler()
         if X_test_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
              test_fill_values = X_train[numerical_cols_to_scale].median()
              X_test_scaled.loc[:, numerical_cols_to_scale] = X_test_scaled[numerical_cols_to_scale].fillna(test_fill_values)
         if X_train_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
             X_train_scaled.loc[:, numerical_cols_to_scale] = X_train_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())
         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_scaled[numerical_cols_to_scale])
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_scaled[numerical_cols_to_scale])
    print("  Scaling appliqué.")
    y_train_final = y_train_resampled

    # 4. Entraînement et Évaluation
    model_configs = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, max_depth=15, min_samples_leaf=5, class_weight=None),
        "LightGBM": lgb.LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1, class_weight=None) if lgbm_available else None
    }
    results_smote = {} # Dictionnaire spécifique pour cette méthode
    for name, model in model_configs.items():
        if model is None: continue
        print(f"\n  --- Modèle : {name} avec {IMBALANCE_METHOD_CURRENT_TEST} ---")
        model.fit(X_train_scaled.fillna(0), y_train_final)
        y_pred = model.predict(X_test_scaled.fillna(0))
        y_proba = model.predict_proba(X_test_scaled.fillna(0))[:, 1]
        print(classification_report(y_test, y_pred, zero_division=0))
        roc_auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        pr_auc = auc(recall, precision)
        report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        print(f"    AUC ROC: {roc_auc:.4f}, AUC PR: {pr_auc:.4f}")
        results_smote[name] = {'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                               'F1_Class1': report_dict.get('1', {}).get('f1-score', 0),
                               'Recall_Class1': report_dict.get('1', {}).get('recall', 0),
                               'Precision_Class1': report_dict.get('1', {}).get('precision', 0)}
    print(f"\n  Résultats pour {IMBALANCE_METHOD_CURRENT_TEST}:")
    print(pd.DataFrame(results_smote).T)
else:
    print("Données sources (X_source, y_source) non chargées. Exécutez la Cellule 1.")
print(f"--- Fin Test {IMBALANCE_METHOD_CURRENT_TEST} ---")
# --- FIN Cellule 2 ---


--- Test avec Méthode d'Équilibrage : SMOTE ---
  Application de SMOTE...
  Shape Train après SMOTE: (1554158, 94)
  Scaling appliqué.

  --- Modèle : RandomForest avec SMOTE ---
              precision    recall  f1-score   support

           0       0.96      0.95      0.96    194271
           1       0.50      0.56      0.53     15986

    accuracy                           0.92    210257
   macro avg       0.73      0.76      0.74    210257
weighted avg       0.93      0.92      0.93    210257

    AUC ROC: 0.9204, AUC PR: 0.5459

  --- Modèle : LightGBM avec SMOTE ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 777079, number of negative: 777079
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.554187 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9865
[Lig

In [3]:
# =============================================================================
# CELLULE 3 : TEST AVEC ADASYN
# =============================================================================
if data_loaded_successfully:
    print("\n--- Test avec Méthode d'Équilibrage : ADASYN ---")
    IMBALANCE_METHOD_CURRENT_TEST = 'ADASYN'

    # 1. Séparation Train/Test
    X_train, X_test, y_train, y_test = train_test_split(X_source, y_source, test_size=0.2,
                                                        random_state=RANDOM_STATE, stratify=y_source)
    # 2. Rééquilibrage ADASYN
    print(f"  Application de {IMBALANCE_METHOD_CURRENT_TEST}...")
    sampler = ADASYN(random_state=RANDOM_STATE)
    X_train_filled = X_train.fillna(X_train.median())
    X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_filled, y_train)
    print(f"  Shape Train après {IMBALANCE_METHOD_CURRENT_TEST}: {X_train_resampled.shape}")

    # 3. Scaling
    # ... (Copier le bloc de scaling de la Cellule 2) ...
    numerical_cols_to_scale = X_source.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X_source.select_dtypes(include='int').columns if X_source[col].nunique(dropna=False) <= 2]
    age_group_cols = [col for col in X_source.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in numerical_cols_to_scale if c not in cols_to_exclude_from_scaling and c in X_train_resampled.columns]
    X_train_scaled = X_train_resampled.copy()
    X_test_scaled = X_test.copy()
    if numerical_cols_to_scale:
         scaler = StandardScaler()
         if X_test_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
              test_fill_values = X_train[numerical_cols_to_scale].median()
              X_test_scaled.loc[:, numerical_cols_to_scale] = X_test_scaled[numerical_cols_to_scale].fillna(test_fill_values)
         if X_train_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
             X_train_scaled.loc[:, numerical_cols_to_scale] = X_train_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())
         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_scaled[numerical_cols_to_scale])
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_scaled[numerical_cols_to_scale])
    print("  Scaling appliqué.")
    y_train_final = y_train_resampled

    # 4. Entraînement et Évaluation
    # ... (Copier le bloc d'entraînement/évaluation de la Cellule 2,
    #      mais changer le nom du dictionnaire de résultats en results_adasyn) ...
    model_configs = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, max_depth=15, min_samples_leaf=5, class_weight=None),
        "LightGBM": lgb.LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1, class_weight=None) if lgbm_available else None
    }
    results_adasyn = {} # Nom du dictionnaire changé
    for name, model in model_configs.items():
        if model is None: continue
        print(f"\n  --- Modèle : {name} avec {IMBALANCE_METHOD_CURRENT_TEST} ---")
        model.fit(X_train_scaled.fillna(0), y_train_final)
        y_pred = model.predict(X_test_scaled.fillna(0))
        y_proba = model.predict_proba(X_test_scaled.fillna(0))[:, 1]
        print(classification_report(y_test, y_pred, zero_division=0))
        roc_auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        pr_auc = auc(recall, precision)
        report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        print(f"    AUC ROC: {roc_auc:.4f}, AUC PR: {pr_auc:.4f}")
        results_adasyn[name] = {'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                               'F1_Class1': report_dict.get('1', {}).get('f1-score', 0),
                               'Recall_Class1': report_dict.get('1', {}).get('recall', 0),
                               'Precision_Class1': report_dict.get('1', {}).get('precision', 0)}
    print(f"\n  Résultats pour {IMBALANCE_METHOD_CURRENT_TEST}:")
    print(pd.DataFrame(results_adasyn).T)
else:
    print("Données sources (X_source, y_source) non chargées. Exécutez la Cellule 1.")
print(f"--- Fin Test {IMBALANCE_METHOD_CURRENT_TEST} ---")
# --- FIN Cellule 3 ---


--- Test avec Méthode d'Équilibrage : ADASYN ---
  Application de ADASYN...
  Shape Train après ADASYN: (1540060, 94)
  Scaling appliqué.

  --- Modèle : RandomForest avec ADASYN ---
              precision    recall  f1-score   support

           0       0.96      0.95      0.96    194271
           1       0.49      0.56      0.52     15986

    accuracy                           0.92    210257
   macro avg       0.73      0.76      0.74    210257
weighted avg       0.93      0.92      0.92    210257

    AUC ROC: 0.9195, AUC PR: 0.5397

  --- Modèle : LightGBM avec ADASYN ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 762981, number of negative: 777079
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.279163 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9837

In [4]:
# =============================================================================
# CELLULE 4 : TEST AVEC SMOTETomek
# =============================================================================
if data_loaded_successfully:
    print("\n--- Test avec Méthode d'Équilibrage : SMOTETomek ---")
    IMBALANCE_METHOD_CURRENT_TEST = 'SMOTETomek'

    # 1. Séparation Train/Test
    X_train, X_test, y_train, y_test = train_test_split(X_source, y_source, test_size=0.2,
                                                        random_state=RANDOM_STATE, stratify=y_source)
    # 2. Rééquilibrage SMOTETomek
    print(f"  Application de {IMBALANCE_METHOD_CURRENT_TEST} (peut être long)...")
    sampler = SMOTETomek(random_state=RANDOM_STATE) # n_jobs n'est pas un argument standard ici
    X_train_filled = X_train.fillna(X_train.median())
    start_time_resample = datetime.now()
    X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_filled, y_train)
    resample_time = datetime.now() - start_time_resample
    print(f"  {IMBALANCE_METHOD_CURRENT_TEST} terminé en {resample_time}")
    print(f"  Shape Train après {IMBALANCE_METHOD_CURRENT_TEST}: {X_train_resampled.shape}")

    # 3. Scaling
    # ... (Copier le bloc de scaling de la Cellule 2) ...
    numerical_cols_to_scale = X_source.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X_source.select_dtypes(include='int').columns if X_source[col].nunique(dropna=False) <= 2]
    age_group_cols = [col for col in X_source.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in numerical_cols_to_scale if c not in cols_to_exclude_from_scaling and c in X_train_resampled.columns]
    X_train_scaled = X_train_resampled.copy()
    X_test_scaled = X_test.copy()
    if numerical_cols_to_scale:
         scaler = StandardScaler()
         if X_test_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
              test_fill_values = X_train[numerical_cols_to_scale].median()
              X_test_scaled.loc[:, numerical_cols_to_scale] = X_test_scaled[numerical_cols_to_scale].fillna(test_fill_values)
         if X_train_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
             X_train_scaled.loc[:, numerical_cols_to_scale] = X_train_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())

         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_scaled[numerical_cols_to_scale])
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_scaled[numerical_cols_to_scale])
    print("  Scaling appliqué.")
    y_train_final = y_train_resampled

    # 4. Entraînement et Évaluation
    # ... (Copier le bloc d'entraînement/évaluation de la Cellule 2,
    #      mais changer le nom du dictionnaire de résultats en results_smotetomek) ...
    model_configs = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, max_depth=15, min_samples_leaf=5, class_weight=None),
        "LightGBM": lgb.LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1, class_weight=None) if lgbm_available else None
    }
    results_smotetomek = {} # Nom du dictionnaire changé
    for name, model in model_configs.items():
        if model is None: continue
        print(f"\n  --- Modèle : {name} avec {IMBALANCE_METHOD_CURRENT_TEST} ---")
        model.fit(X_train_scaled.fillna(0), y_train_final)
        y_pred = model.predict(X_test_scaled.fillna(0))
        y_proba = model.predict_proba(X_test_scaled.fillna(0))[:, 1]
        print(classification_report(y_test, y_pred, zero_division=0))
        roc_auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        pr_auc = auc(recall, precision)
        report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        print(f"    AUC ROC: {roc_auc:.4f}, AUC PR: {pr_auc:.4f}")
        results_smotetomek[name] = {'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                                   'F1_Class1': report_dict.get('1', {}).get('f1-score', 0),
                                   'Recall_Class1': report_dict.get('1', {}).get('recall', 0),
                                   'Precision_Class1': report_dict.get('1', {}).get('precision', 0)}
    print(f"\n  Résultats pour {IMBALANCE_METHOD_CURRENT_TEST}:")
    print(pd.DataFrame(results_smotetomek).T)
else:
    print("Données sources (X_source, y_source) non chargées. Exécutez la Cellule 1.")
print(f"--- Fin Test {IMBALANCE_METHOD_CURRENT_TEST} ---")
# --- FIN Cellule 4 ---


--- Test avec Méthode d'Équilibrage : SMOTETomek ---
  Application de SMOTETomek (peut être long)...
  SMOTETomek terminé en 5:39:53.495618
  Shape Train après SMOTETomek: (1548912, 94)
  Scaling appliqué.

  --- Modèle : RandomForest avec SMOTETomek ---
              precision    recall  f1-score   support

           0       0.96      0.95      0.96    194271
           1       0.50      0.57      0.53     15986

    accuracy                           0.92    210257
   macro avg       0.73      0.76      0.74    210257
weighted avg       0.93      0.92      0.93    210257

    AUC ROC: 0.9205, AUC PR: 0.5465

  --- Modèle : LightGBM avec SMOTETomek ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 774456, number of negative: 774456
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.523230 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not 

In [5]:
# =============================================================================
# CELLULE 5 : TEST AVEC RandomUnderSampler
# =============================================================================
if data_loaded_successfully:
    print("\n--- Test avec Méthode d'Équilibrage : RandomUnderSampler ---")
    IMBALANCE_METHOD_CURRENT_TEST = 'RandomUnderSampler'

    # 1. Séparation Train/Test
    X_train, X_test, y_train, y_test = train_test_split(X_source, y_source, test_size=0.2,
                                                        random_state=RANDOM_STATE, stratify=y_source)
    # 2. Rééquilibrage RandomUnderSampler
    print(f"  Application de {IMBALANCE_METHOD_CURRENT_TEST}...")
    sampler = RandomUnderSampler(random_state=RANDOM_STATE)
    X_train_filled = X_train.fillna(X_train.median())
    X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_filled, y_train)
    print(f"  Shape Train après {IMBALANCE_METHOD_CURRENT_TEST}: {X_train_resampled.shape}") # Sera plus petit

    # 3. Scaling
    # ... (Copier le bloc de scaling) ...
    numerical_cols_to_scale = X_source.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X_source.select_dtypes(include='int').columns if X_source[col].nunique(dropna=False) <= 2]
    age_group_cols = [col for col in X_source.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in numerical_cols_to_scale if c not in cols_to_exclude_from_scaling and c in X_train_resampled.columns]
    X_train_scaled = X_train_resampled.copy()
    X_test_scaled = X_test.copy()
    if numerical_cols_to_scale:
         scaler = StandardScaler()
         if X_test_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
              test_fill_values = X_train[numerical_cols_to_scale].median()
              X_test_scaled.loc[:, numerical_cols_to_scale] = X_test_scaled[numerical_cols_to_scale].fillna(test_fill_values)
         if X_train_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
             X_train_scaled.loc[:, numerical_cols_to_scale] = X_train_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())

         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_scaled[numerical_cols_to_scale])
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_scaled[numerical_cols_to_scale])
    print("  Scaling appliqué.")
    y_train_final = y_train_resampled

    # 4. Entraînement et Évaluation
    # ... (Copier le bloc d'entraînement/évaluation,
    #      mais changer le nom du dictionnaire de résultats en results_rus) ...
    model_configs = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, max_depth=15, min_samples_leaf=5, class_weight=None),
        "LightGBM": lgb.LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1, class_weight=None) if lgbm_available else None
    }
    results_rus = {} # Nom du dictionnaire changé
    for name, model in model_configs.items():
        if model is None: continue
        print(f"\n  --- Modèle : {name} avec {IMBALANCE_METHOD_CURRENT_TEST} ---")
        model.fit(X_train_scaled.fillna(0), y_train_final)
        y_pred = model.predict(X_test_scaled.fillna(0))
        y_proba = model.predict_proba(X_test_scaled.fillna(0))[:, 1]
        print(classification_report(y_test, y_pred, zero_division=0))
        roc_auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        pr_auc = auc(recall, precision)
        report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        print(f"    AUC ROC: {roc_auc:.4f}, AUC PR: {pr_auc:.4f}")
        results_rus[name] = {'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                            'F1_Class1': report_dict.get('1', {}).get('f1-score', 0),
                            'Recall_Class1': report_dict.get('1', {}).get('recall', 0),
                            'Precision_Class1': report_dict.get('1', {}).get('precision', 0)}
    print(f"\n  Résultats pour {IMBALANCE_METHOD_CURRENT_TEST}:")
    print(pd.DataFrame(results_rus).T)
else:
    print("Données sources (X_source, y_source) non chargées. Exécutez la Cellule 1.")
print(f"--- Fin Test {IMBALANCE_METHOD_CURRENT_TEST} ---")
# --- FIN Cellule 5 ---


--- Test avec Méthode d'Équilibrage : RandomUnderSampler ---
  Application de RandomUnderSampler...
  Shape Train après RandomUnderSampler: (127890, 94)
  Scaling appliqué.

  --- Modèle : RandomForest avec RandomUnderSampler ---
              precision    recall  f1-score   support

           0       0.99      0.80      0.89    194271
           1       0.27      0.88      0.41     15986

    accuracy                           0.81    210257
   macro avg       0.63      0.84      0.65    210257
weighted avg       0.93      0.81      0.85    210257

    AUC ROC: 0.9241, AUC PR: 0.5627

  --- Modèle : LightGBM avec RandomUnderSampler ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 63945, number of negative: 63945
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.166477 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set

In [6]:
# =============================================================================
# CELLULE 6 : TEST AVEC CLASS_WEIGHT
# =============================================================================
if data_loaded_successfully:
    print("\n--- Test avec Méthode d'Équilibrage : CLASS_WEIGHT ---")
    IMBALANCE_METHOD_CURRENT_TEST = 'CLASS_WEIGHT'

    # 1. Séparation Train/Test
    X_train, X_test, y_train, y_test = train_test_split(X_source, y_source, test_size=0.2,
                                                        random_state=RANDOM_STATE, stratify=y_source)
    # 2. PAS de Rééquilibrage ici
    print("  Aucun resampling appliqué pour CLASS_WEIGHT.")
    X_train_processed = X_train.copy() # Utiliser données d'entraînement originales
    y_train_processed = y_train.copy()

    # 3. Scaling
    # ... (Copier le bloc de scaling) ...
    numerical_cols_to_scale = X_source.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X_source.select_dtypes(include='int').columns if X_source[col].nunique(dropna=False) <= 2]
    age_group_cols = [col for col in X_source.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in numerical_cols_to_scale if c not in cols_to_exclude_from_scaling and c in X_train_processed.columns]
    X_train_scaled = X_train_processed.copy()
    X_test_scaled = X_test.copy()
    if numerical_cols_to_scale:
         scaler = StandardScaler()
         if X_test_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
              test_fill_values = X_train[numerical_cols_to_scale].median()
              X_test_scaled.loc[:, numerical_cols_to_scale] = X_test_scaled[numerical_cols_to_scale].fillna(test_fill_values)
         if X_train_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
             X_train_scaled.loc[:, numerical_cols_to_scale] = X_train_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())

         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_scaled[numerical_cols_to_scale])
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_scaled[numerical_cols_to_scale])
    print("  Scaling appliqué.")
    y_train_final = y_train_processed


    # 4. Entraînement et Évaluation (AVEC class_weight='balanced')
    model_configs_cw = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1,
                                             max_depth=15, min_samples_leaf=5, class_weight='balanced'), # MODIFIÉ
        "LightGBM": lgb.LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1,
                                     class_weight='balanced') if lgbm_available else None # MODIFIÉ
    }
    results_classweight = {} # Nom du dictionnaire changé
    for name, model in model_configs_cw.items():
        if model is None: continue
        print(f"\n  --- Modèle : {name} avec {IMBALANCE_METHOD_CURRENT_TEST} ---")
        model.fit(X_train_scaled.fillna(0), y_train_final)
        y_pred = model.predict(X_test_scaled.fillna(0))
        y_proba = model.predict_proba(X_test_scaled.fillna(0))[:, 1]
        print(classification_report(y_test, y_pred, zero_division=0))
        roc_auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        pr_auc = auc(recall, precision)
        report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        print(f"    AUC ROC: {roc_auc:.4f}, AUC PR: {pr_auc:.4f}")
        results_classweight[name] = {'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                                   'F1_Class1': report_dict.get('1', {}).get('f1-score', 0),
                                   'Recall_Class1': report_dict.get('1', {}).get('recall', 0),
                                   'Precision_Class1': report_dict.get('1', {}).get('precision', 0)}
    print(f"\n  Résultats pour {IMBALANCE_METHOD_CURRENT_TEST}:")
    print(pd.DataFrame(results_classweight).T)
else:
    print("Données sources (X_source, y_source) non chargées. Exécutez la Cellule 1.")
print(f"--- Fin Test {IMBALANCE_METHOD_CURRENT_TEST} ---")
# --- FIN Cellule 6 ---


--- Test avec Méthode d'Équilibrage : CLASS_WEIGHT ---
  Aucun resampling appliqué pour CLASS_WEIGHT.
  Scaling appliqué.

  --- Modèle : RandomForest avec CLASS_WEIGHT ---
              precision    recall  f1-score   support

           0       0.98      0.86      0.92    194271
           1       0.33      0.81      0.46     15986

    accuracy                           0.86    210257
   macro avg       0.65      0.84      0.69    210257
weighted avg       0.93      0.86      0.88    210257

    AUC ROC: 0.9235, AUC PR: 0.5658

  --- Modèle : LightGBM avec CLASS_WEIGHT ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 63945, number of negative: 777079
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.117534 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7961
[Lig

In [7]:
# =============================================================================
# CELLULE 8 : COMPARAISON FINALE DES MÉTHODES D'ÉQUILIBRAGE (SANS SMOTEENN)
# =============================================================================
print("\n--- Tableau Comparatif Final des Méthodes d'Équilibrage ---")

all_benchmark_results_list = []

# Assurez-vous que les dictionnaires de résultats existent et correspondent aux noms utilisés
if 'results_smote' in locals(): all_benchmark_results_list.append(pd.DataFrame(results_smote).T.assign(Balancing_Method='SMOTE'))
if 'results_adasyn' in locals(): all_benchmark_results_list.append(pd.DataFrame(results_adasyn).T.assign(Balancing_Method='ADASYN'))
if 'results_smotetomek' in locals(): all_benchmark_results_list.append(pd.DataFrame(results_smotetomek).T.assign(Balancing_Method='SMOTETomek'))
if 'results_rus' in locals(): all_benchmark_results_list.append(pd.DataFrame(results_rus).T.assign(Balancing_Method='RandomUnderSampler'))
if 'results_classweight' in locals(): all_benchmark_results_list.append(pd.DataFrame(results_classweight).T.assign(Balancing_Method='CLASS_WEIGHT'))
if 'results_none' in locals(): all_benchmark_results_list.append(pd.DataFrame(results_none).T.assign(Balancing_Method='NONE'))
# Si vous avez gardé les résultats CTGAN d'une exécution précédente et les avez chargés :
# if 'results_gan_df' in locals() and isinstance(results_gan_df, pd.DataFrame):
#    results_gan_df['Balancing_Method'] = 'CTGAN'
#    results_gan_df = results_gan_df.reset_index().rename(columns={'index': 'Model'})
#    all_benchmark_results_list.append(results_gan_df)


if all_benchmark_results_list:
    final_comparison_table = pd.concat(all_benchmark_results_list)
    if 'Model' not in final_comparison_table.columns: # S'assurer que 'Model' existe après concat
         final_comparison_table = final_comparison_table.reset_index().rename(columns={'index': 'Model'})

    # Trier par AUC PR descendant pour chaque modèle, puis par Modèle
    final_comparison_table = final_comparison_table.sort_values(by=['Model', 'AUC_PR'], ascending=[True, False])
    pd.options.display.float_format = '{:.4f}'.format
    print(final_comparison_table[['Model', 'Balancing_Method', 'AUC_PR', 'F1_Class1', 'Recall_Class1', 'Precision_Class1', 'AUC_ROC']])
else:
    print("Aucun résultat de benchmark à afficher.")
# --- FIN Cellule 8 ---


--- Tableau Comparatif Final des Méthodes d'Équilibrage ---
          Model    Balancing_Method  AUC_PR  F1_Class1  Recall_Class1  \
9      LightGBM        CLASS_WEIGHT  0.5915     0.4360         0.8723   
7      LightGBM  RandomUnderSampler  0.5888     0.4325         0.8727   
1      LightGBM               SMOTE  0.5801     0.4899         0.3836   
5      LightGBM          SMOTETomek  0.5793     0.4917         0.3853   
3      LightGBM              ADASYN  0.5787     0.4874         0.3785   
8  RandomForest        CLASS_WEIGHT  0.5658     0.4643         0.8112   
6  RandomForest  RandomUnderSampler  0.5627     0.4135         0.8809   
4  RandomForest          SMOTETomek  0.5465     0.5295         0.5685   
0  RandomForest               SMOTE  0.5459     0.5279         0.5609   
2  RandomForest              ADASYN  0.5397     0.5239         0.5589   

   Precision_Class1  AUC_ROC  
9            0.2906   0.9291  
7            0.2875   0.9285  
1            0.6778   0.9258  
5          